In [1]:
import pandas as pd 
import numpy as np 

In [2]:
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
order_items = pd.read_csv('order_items.csv', parse_dates=['shipping_limit_date'])
order_payments = pd.read_csv('order_payments.csv')
order_reviews = pd.read_csv('order_reviews.csv')
products = pd.read_csv('products.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')
sellers = pd.read_csv('sellers.csv')
geolocation = pd.read_csv('geolocation.csv')

In [3]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
print(customers.shape)
customers.head()
customers.info()

(99441, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [6]:
print("Duplicate customer_id:", customers['customer_id'].duplicated().sum())
print("Fully duplicate rows:", customers.duplicated().sum())

Duplicate customer_id: 0
Fully duplicate rows: 0


In [7]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [8]:
print("Total rows:", len(customers))
print("Unique customer_id:", customers['customer_id'].nunique())
print("Unique customer_unique_id:", customers['customer_unique_id'].nunique())

Total rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096


In [10]:
print("Unique states:", customers['customer_state'].nunique())
print(customers['customer_state'].value_counts().head(10))

print("Unique cities:", customers['customer_city'].nunique())

Unique states: 27
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64
Unique cities: 4119


In [11]:
print(customers['customer_zip_code_prefix'].dtype)
customers['customer_zip_code_prefix'].head()

int64


0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: int64

In [12]:
customers['customer_city'] = customers['customer_city'].str.strip().str.title()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()

In [13]:
print(f"Final customers table: {len(customers):,} rows")
print(f"Unique customer_id: {customers['customer_id'].nunique():,}")
print(f"Unique customer_unique_id: {customers['customer_unique_id'].nunique():,}")
customers.isna().sum()

Final customers table: 99,441 rows
Unique customer_id: 99,441
Unique customer_unique_id: 96,096


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [14]:
customers.to_csv('customers_clean.csv', index=False)

In [15]:
print(orders.shape)
orders.head()
orders.info()

(99441, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [16]:
print("Duplicate order_id:", orders['order_id'].duplicated().sum())
print("Fully duplicate rows:", orders.duplicated().sum())

Duplicate order_id: 0
Fully duplicate rows: 0


In [17]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [18]:
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [19]:
print(orders['order_purchase_timestamp'].min())
print(orders['order_purchase_timestamp'].max())

print(orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index())

2016-09-04 21:15:19
2018-10-17 17:30:18
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


In [20]:
print("Rows in orders:", len(orders))
print("Unique customer_id in orders:", orders['customer_id'].nunique())

Rows in orders: 99441
Unique customer_id in orders: 99441


In [21]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(delivered):,}")
print(f"Excluded (non-delivered): {len(orders) - len(delivered):,}")
delivered.isna().sum()

Delivered orders: 96,478
Excluded (non-delivered): 2,963


order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [22]:
delivered = delivered.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)
print("Nulls after merge:", delivered['customer_unique_id'].isna().sum())
print(f"Rows: {len(delivered):,}")

Nulls after merge: 0
Rows: 96,478


In [23]:
delivered.isna().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
customer_unique_id                0
customer_city                     0
customer_state                    0
dtype: int64

In [24]:
delivered.to_csv('orders_clean.csv', index=False)

In [25]:
print(order_items.shape)
order_items.head()
order_items.info()

(112650, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [26]:
print("Duplicate rows:", order_items.duplicated().sum())

Duplicate rows: 0


In [27]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [28]:
print("Rows with price <= 0:", (order_items['price'] <= 0).sum())
print("Rows with freight_value < 0:", (order_items['freight_value'] < 0).sum())
print("Price range:", order_items['price'].min(), "to", order_items['price'].max())

Rows with price <= 0: 0
Rows with freight_value < 0: 0
Price range: 0.85 to 6735.0


In [29]:
order_item_rev = order_items.groupby('order_id').agg(
    item_revenue=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()

In [30]:
orders_final = delivered.merge(order_item_rev, on='order_id', how='left')
orders_final['order_revenue'] = orders_final['item_revenue']

In [31]:
orders_final = orders_final.dropna(subset=['order_revenue'])

In [33]:

print(f"Rows after dropna: {len(orders_final):,}")

Rows after dropna: 96,478


In [79]:
print(f"Final orders table: {len(orders_final):,} orders")
print(f"Unique customers: {orders_final['customer_id'].nunique():,}")
print(f"Total revenue: ${orders_final['order_revenue'].sum():,.2f}")
print(f"Average Order Value: ${orders_final['order_revenue'].sum() / len(orders_final):.2f}")

Final orders table: 96,478 orders
Unique customers: 96,478
Total revenue: $13,221,498.11
Average Order Value: $137.04


In [36]:
print(order_payments.shape)
order_payments.head()
order_payments.info()

(103886, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [37]:
print("Duplicate rows:", order_payments.duplicated().sum())
order_payments.isna().sum()

Duplicate rows: 0


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [38]:
print(order_payments['payment_type'].value_counts())
print("Installments range:", order_payments['payment_installments'].min(), "to", order_payments['payment_installments'].max())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
Installments range: 0 to 24


In [39]:
print("Rows with payment_value <= 0:", (order_payments['payment_value'] <= 0).sum())
print("Rows with 'not_defined' payment_type:", (order_payments['payment_type'] == 'not_defined').sum())

Rows with payment_value <= 0: 9
Rows with 'not_defined' payment_type: 3


In [40]:
payments_per_order = order_payments.groupby('order_id').size()
print("Orders with 1 payment row:", (payments_per_order == 1).sum())
print("Orders with 2+ payment rows:", (payments_per_order > 1).sum())

Orders with 1 payment row: 96479
Orders with 2+ payment rows: 2961


In [45]:
order_payments_agg = order_payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_methods_used=('payment_type', 'nunique'),
    max_installments=('payment_installments', 'max')
).reset_index()

print(order_payments_agg.head())

                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   
3  00024acbcdf0a6daa1e931b038114c75                25.78   
4  00042b26cf59d7ce69dfabb4e55b4fd9               218.04   

   payment_methods_used  max_installments  
0                     1                 2  
1                     1                 3  
2                     1                 5  
3                     1                 2  
4                     1                 3  


In [46]:
check = orders_final.merge(order_payments_agg, on='order_id', how='left')
check['payment_vs_revenue_diff'] = check['total_payment_value'] - (check['order_revenue'] + check['freight_value'])
print(check['payment_vs_revenue_diff'].describe())

count    96477.000000
mean         0.029349
std          1.138706
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_vs_revenue_diff, dtype: float64


In [47]:
order_payments_agg.to_csv('order_payments_clean.csv', index=False)

In [48]:
print(order_reviews.shape)
order_reviews.head()
order_reviews.info()

(99224, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [51]:
print(f"Fully identical duplicate rows: {order_reviews.duplicated().sum()}")

Fully identical duplicate rows: 0


In [52]:
  order_reviews = order_reviews.drop_duplicates()
  print(f"Rows after dropping exact duplicates: {len(order_reviews):,}")

Rows after dropping exact duplicates: 99,224


In [53]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [54]:
order_reviews['review_comment_message']=order_reviews['review_comment_message'].fillna('No comment')

In [55]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,No comment,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,No comment,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,No comment,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [56]:
order_reviews['review_comment_title']=order_reviews['review_comment_title'].fillna('No Title')

In [59]:
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

In [60]:
print(f"Final order_reviews: {len(order_reviews):,} rows")
print(f"Unique order_id: {order_reviews['order_id'].nunique():,}")
order_reviews.isna().sum()

Final order_reviews: 99,224 rows
Unique order_id: 98,673


review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [61]:
order_reviews.to_csv('order_reviews_clean.csv', index=False)

In [62]:
print(products.shape)
products.head()
products.info()

(32951, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [63]:
print("Duplicate product_id:", products['product_id'].duplicated().sum())
print("Fully duplicate rows:", products.duplicated().sum())

Duplicate product_id: 0
Fully duplicate rows: 0


In [64]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [65]:
products = products.merge(category_translation, on='product_category_name', how='left')
print("Missing category after merge:", products['product_category_name_english'].isna().sum())

Missing category after merge: 623


In [66]:
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

In [67]:
print("Number of unique categories:", products['product_category_name_english'].nunique())
print(products['product_category_name_english'].value_counts().head(10))

Number of unique categories: 72
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [68]:
print(f"Final products table: {len(products):,} rows")
products.isna().sum()

Final products table: 32,951 rows


product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english      0
dtype: int64

In [69]:
products.to_csv('products_clean.csv', index=False)

In [70]:
print(sellers.shape)
sellers.head()
sellers.info()

(3095, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [71]:
print("Duplicate seller_id:", sellers['seller_id'].duplicated().sum())
sellers.isna().sum()

Duplicate seller_id: 0


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [72]:
sellers.to_csv('sellers_clean.csv', index=False)

In [73]:
from sqlalchemy import create_engine


username = "root"
password = "Nihar%402031"   
host = "localhost"
port = "3306"
database = "ecommerce"
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")


customers.to_sql("customers", engine, if_exists="replace", index=False)
orders_final.to_sql("orders", engine, if_exists="replace", index=False)
order_items.to_sql("order_items", engine, if_exists="replace", index=False)
order_payments_agg.to_sql("order_payments", engine, if_exists="replace", index=False)
order_reviews.to_sql("order_reviews", engine, if_exists="replace", index=False)
products.to_sql("products", engine, if_exists="replace", index=False)
sellers.to_sql("sellers", engine, if_exists="replace", index=False)
cust_orders.to_sql("customer_summary", engine, if_exists="replace", index=False)

NameError: name 'cust_orders' is not defined

In [74]:
cust_orders = orders_final.groupby('customer_unique_id').agg(
    n_orders=('order_id', 'nunique'),
    total_spend=('order_revenue', 'sum'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max')
).reset_index()

cust_orders['is_repeat'] = cust_orders['n_orders'] > 1

print(cust_orders.head())
print(f"Total unique customers: {len(cust_orders):,}")

                 customer_unique_id  n_orders  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2         1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f         1        18.90   
2  0000f46a3911fa3c0805444483337064         1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078         1        25.99   
4  0004aac84e0df4da2b147fca70cf8255         1       180.00   

          first_order          last_order  is_repeat  
0 2018-05-10 10:56:27 2018-05-10 10:56:27      False  
1 2018-05-07 11:11:27 2018-05-07 11:11:27      False  
2 2017-03-10 21:05:03 2017-03-10 21:05:03      False  
3 2017-10-12 20:29:41 2017-10-12 20:29:41      False  
4 2017-11-14 19:45:42 2017-11-14 19:45:42      False  
Total unique customers: 93,358


In [75]:
cust_orders.to_sql("customer_summary", engine, if_exists="replace", index=False)

93358

In [76]:
orders.to_sql("orders_all", engine, if_exists="replace", index=False)

99441

In [82]:

orders_all = orders.merge(
    customers[
        ['customer_id',
         'customer_unique_id',
         'customer_city',
         'customer_state']
    ],
    on='customer_id',
    how='left'
)

In [83]:
orders_all[['order_id', 'customer_id', 'customer_unique_id']].head()

,order_id,customer_id,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6


In [84]:
customer_summary_all = orders_all.groupby('customer_unique_id').agg(
    n_orders=('order_id', 'nunique'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max')
).reset_index()

customer_summary_all['is_repeat'] = (
    customer_summary_all['n_orders'] > 1
)

customer_summary_all.head()

,customer_unique_id,n_orders,first_order,last_order,is_repeat
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,False
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,False
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,False
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,False


In [85]:
print("Total unique customers:",
      customer_summary_all['customer_unique_id'].nunique())

print("Repeat customers:",
      customer_summary_all['is_repeat'].sum())

print("One-time customers:",
      (~customer_summary_all['is_repeat']).sum())

Total unique customers: 96096
Repeat customers: 2997
One-time customers: 93099


In [88]:
customer_summary_all.to_sql(
    "customer_summary_all",
    engine,
    if_exists="replace",
    index=False
)

print("customer_summary_all uploaded successfully.")

customer_summary_all uploaded successfully.


In [89]:
geolocation = pd.read_csv('geolocation.csv')

In [91]:
engine.dispose()

In [92]:
geolocation.to_sql("geolocation",
    engine,
    if_exists="replace",
    index=False
)

1000163